# Práctica 16 — Detección y Análisis de Rostros en Vivo
**Machine Learning · ISTER 2026 · Ing. David Minango, PhD**

---

**Objetivo:** Habilitar tu propia cámara dentro de Google Colab, preprocesar la foto capturada, detectar el rostro con un clasificador clásico de OpenCV (Haar Cascade), y estimar edad y sexo con un modelo ya entrenado (InsightFace).

Completa las celdas marcadas con `# 🔧 TU CÓDIGO`. Las demás celdas ya están listas — solo ejecútalas.

⚠️ **Importante:** la edad y el sexo que reporta InsightFace son ESTIMACIONES basadas en apariencia, no un dato verificado. Úsalo con espíritu crítico.

## Parte 0 — Setup (solo ejecutar)

In [ ]:
!pip install insightface onnxruntime -q

import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

print("✅ Librerías listas")

## Parte 1 — Habilitar la Cámara y Capturar una Foto

Colab corre en un servidor en la nube, no tiene cámara propia. Este código en JavaScript le pide prestada la cámara a TU navegador, toma la foto, y se la entrega a Python.

In [ ]:
# Celda 1.1 — Dado: función para capturar una foto con tu cámara
def tomar_foto(nombre_archivo='mi_foto.jpg', calidad=0.8):
    js = Javascript('''
    async function tomarFoto(calidad) {
      const div = document.createElement('div');
      const boton = document.createElement('button');
      boton.textContent = '📸 Capturar foto';
      div.appendChild(boton);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      await new Promise((resolve) => boton.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', calidad);
    }
    ''')
    display(js)
    datos = eval_js('tomarFoto({})'.format(calidad))
    binario = b64decode(datos.split(',')[1])
    with open(nombre_archivo, 'wb') as f:
        f.write(binario)
    return nombre_archivo

print("✅ Función tomar_foto() lista")

In [ ]:
# Celda 1.2 — Dado: capturar (o subir) tu foto
# Da clic en "Permitir" cuando el navegador pida acceso a la cámara,
# y luego en el botón "📸 Capturar foto" que aparece en la salida de esta celda.
archivo = tomar_foto()

# Alternativa si tu navegador no da acceso a la cámara:
# from google.colab import files
# subido = files.upload()
# archivo = list(subido.keys())[0]

imagen = cv2.imread(archivo)
print("Foto capturada, forma:", imagen.shape)

**❓ Antes de continuar:**
1. ¿Por qué Colab no puede simplemente usar `cv2.VideoCapture(0)` como en un script normal de Python?
2. ¿Qué tipo de dato es `datos` justo antes de `b64decode`? (pista: no es la imagen todavía, es texto)

_Responde aquí en esta celda Markdown._

## Parte 2 — Preprocesamiento

In [ ]:
# Celda 2.1 — Tu código: convertir colores y pasar a escala de grises
# 🔧 TU CÓDIGO
imagen_rgb  = ___________          # cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB) — para MOSTRAR bien los colores
imagen_gris = ___________          # cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY) — Haar Cascade necesita escala de grises

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].imshow(imagen_rgb);  axes[0].set_title('Original (RGB)');      axes[0].axis('off')
axes[1].imshow(imagen_gris, cmap='gray'); axes[1].set_title('Escala de grises'); axes[1].axis('off')
plt.show()

**❓ Sobre el preprocesamiento:**
1. ¿Por qué convertimos a RGB solo para MOSTRAR la imagen, pero usamos la versión en escala de grises para detectar?
2. ¿Qué forma (`.shape`) tiene `imagen_gris` comparada con `imagen_rgb`? ¿Por qué son distintas?

_Responde aquí en esta celda Markdown._

## Parte 3 — Detección con Haar Cascade

In [ ]:
# Celda 3.1 — Dado: cargar el clasificador ya entrenado
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
print("✅ Clasificador Haar Cascade cargado")

In [ ]:
# Celda 3.2 — Tu código: detectar y dibujar los rostros
# 🔧 TU CÓDIGO
caras = face_cascade.detectMultiScale(___________, scaleFactor=1.1, minNeighbors=5)   # imagen_gris

resultado = imagen_rgb.copy()
for (x, y, w, h) in caras:
    cv2.rectangle(___________, (x, y), (___________, ___________), (0, 255, 0), 3)   # resultado, (x + w, y + h)

plt.imshow(resultado)
plt.axis('off')
plt.title(f"{len(caras)} rostro(s) detectado(s) con Haar Cascade")
plt.show()

**❓ Sobre la detección:**
1. Prueba bajar `minNeighbors` a 2. ¿Aparecen más o menos rectángulos? ¿Alguno es un falso positivo (marca algo que no es una cara)?
2. Si en tu foto hay más de una persona, ¿el bucle `for` dibuja un rectángulo por cada una? ¿Cómo lo sabes?

_Responde aquí en esta celda Markdown._

## Parte 4 — Edad y Sexo con InsightFace

In [ ]:
# Celda 4.1 — Tu código: cargar el modelo ya entrenado
# 🔧 TU CÓDIGO
from insightface.app import FaceAnalysis

analizador = ___________          # FaceAnalysis(name='buffalo_l', providers=['CPUExecutionProvider'])
___________                       # analizador.prepare(ctx_id=0, det_size=(640, 640))

print("✅ InsightFace listo")

⚠️ La primera vez que corres esta celda, InsightFace descarga el modelo `buffalo_l` (varios cientos de MB) — puede tardar uno o dos minutos.

In [ ]:
# Celda 4.2 — Tu código: analizar la foto y dibujar el resultado
# 🔧 TU CÓDIGO
# OJO: aquí se usa la imagen BGR original de cv2.imread, NO la versión RGB
detecciones = analizador.get(___________)          # imagen

resultado2 = imagen_rgb.copy()
for cara in detecciones:
    x1, y1, x2, y2 = cara.bbox.astype(int)
    edad = int(___________)                        # cara.age
    sexo = 'Hombre' if ___________ == 1 else 'Mujer'   # cara.gender

    cv2.rectangle(resultado2, (x1, y1), (x2, y2), (0, 200, 0), 2)
    cv2.putText(resultado2, f"{sexo}, ~{edad}", (x1, max(y1 - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 200, 0), 2)

plt.imshow(resultado2)
plt.axis('off')
plt.title(f"{len(detecciones)} rostro(s) analizado(s) con InsightFace")
plt.show()

for i, cara in enumerate(detecciones):
    sexo = 'Hombre' if cara.gender == 1 else 'Mujer'
    print(f"Cara {i+1}: sexo estimado = {sexo}, edad estimada = {int(cara.age)} años, "
          f"confianza de detección = {cara.det_score:.2f}")

**❓ Comparación final:**
1. ¿`len(caras)` (Haar Cascade, Parte 3) y `len(detecciones)` (InsightFace, Parte 4) coinciden en tu foto? Si no, ¿cuál te parece más confiable y por qué?
2. La edad estimada, ¿qué tan cerca está de tu edad real? ¿Qué factores (luz, ángulo, gesto) crees que pudieron influir?
3. `cara.det_score` mide la confianza de que ESO sea una cara — no tiene relación con qué tan segura está la edad o el sexo estimados. ¿Por qué son cosas distintas?

_Responde aquí en esta celda Markdown._

## Entrega

- **Notebook con todas las celdas ejecutadas:** todas las celdas `# 🔧 TU CÓDIGO` completadas, incluyendo tu foto capturada y el resultado de ambos detectores. Preguntas ❓ respondidas en celdas Markdown. Exportar como `.ipynb`.
- **Conclusión integradora (párrafo final):** ¿qué diferencia encontraste entre un detector clásico (Haar Cascade) y un modelo moderno (InsightFace)? Y, considerando que estos modelos aprenden de datos que no siempre representan a todo el mundo por igual, ¿qué cuidados tomarías antes de usar una tecnología así en un sistema real (control de acceso, estadísticas de clientes, etc.)?

📅 Entrega al docente antes de la próxima clase.